# Colab Local LLM Classifier
Runs the classifier stage locally on Colab through a vLLM OpenAI-compatible server and writes classifier-only prediction parquet files to Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/data/llm-gatekeeping'
REPO_URL = 'https://github.com/noamdwc/llm-gatekeeping.git'
REPO_DIR = '/content/llm-gatekeeping'
BRANCH = 'zero_shot_nlp_attack'

SPLIT = 'val'
LIMIT = 5  # Set to None for the full split.
MODEL_ID = 'meta/llama-3.1-8b-instruct'
HF_MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 4096
BATCH_SIZE = 1
CHECKPOINT_EVERY = 50
OUTPUT_SUFFIX = 'colab_local_classifier'

HF_CACHE_DIR = f'{DRIVE_ROOT}/cache/huggingface'
SPLITS_DIR = f'{DRIVE_ROOT}/data/processed/splits'
PREDICTIONS_DIR = f'{DRIVE_ROOT}/data/processed/predictions'
CHECKPOINT_PATH = f'{PREDICTIONS_DIR}/llm_checkpoint_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
OUTPUT_PATH = f'{PREDICTIONS_DIR}/llm_predictions_{SPLIT}_{OUTPUT_SUFFIX}.parquet'
VLLM_LOG_PATH = f'{PREDICTIONS_DIR}/vllm_server_{SPLIT}_{OUTPUT_SUFFIX}.log'
VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'

import os

os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_HUB_CACHE'] = f'{HF_CACHE_DIR}/hub'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE_DIR}/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE_DIR}/datasets'
print(f'Hugging Face cache: {HF_CACHE_DIR}')
print(f'Output path: {OUTPUT_PATH}')

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Using existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "vllm>=0.6.0" "openai>=1.0.0"

In [ ]:
from pathlib import Path
import os
import subprocess
import time
from datetime import datetime, timezone

import requests
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is unavailable. Enable Runtime > Change runtime type > GPU.')

Path(PREDICTIONS_DIR).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

server_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', HF_MODEL_ID,
    '--served-model-name', MODEL_ID,
    '--host', '127.0.0.1',
    '--port', '8000',
    '--tensor-parallel-size', str(TENSOR_PARALLEL_SIZE),
    '--gpu-memory-utilization', str(GPU_MEMORY_UTILIZATION),
    '--max-model-len', str(MAX_MODEL_LEN),
]

server_cmd_preview = 'python -m vllm.entrypoints.openai.api_server'

def tail_log(path: Path, line_count: int = 80) -> str:
    if not path.exists():
        return '<log file does not exist>'
    lines = path.read_text(errors='replace').splitlines()
    if not lines:
        return '<log file is empty>'
    return '\n'.join(lines[-line_count:])

def raise_vllm_startup_error(reason: str) -> None:
    log_tail = tail_log(Path(VLLM_LOG_PATH))
    raise RuntimeError(
        f'{reason}\n'
        f'vLLM command: {" ".join(server_cmd)}\n'
        f'vLLM log path: {VLLM_LOG_PATH}\n'
        f'Last vLLM log lines:\n{log_tail}'
    )

def models_response_serves_model(payload: dict, model_id: str) -> tuple[bool, list[str]]:
    served_ids = []
    data = payload.get('data', []) if isinstance(payload, dict) else []
    for model_info in data:
        if isinstance(model_info, dict):
            served_id = model_info.get('id') or model_info.get('model')
            if served_id is not None:
                served_ids.append(str(served_id))
    if isinstance(payload, dict):
        for key in ('id', 'model'):
            served_id = payload.get(key)
            if served_id is not None:
                served_ids.append(str(served_id))
    return model_id in served_ids, served_ids

print('Starting vLLM:', ' '.join(server_cmd))
print(f'vLLM log path: {VLLM_LOG_PATH}')
vllm_log_file = open(VLLM_LOG_PATH, 'a', buffering=1)
print('\n' + '=' * 100, file=vllm_log_file)
print(f'Started at: {datetime.now(timezone.utc).isoformat()}', file=vllm_log_file)
print(f'Command: {" ".join(server_cmd)}', file=vllm_log_file)
print(f'Model alias: {MODEL_ID}', file=vllm_log_file)
print(f'Hugging Face model: {HF_MODEL_ID}', file=vllm_log_file)
print(f'HF_HOME: {os.environ.get("HF_HOME", "")}', file=vllm_log_file)
vllm_log_file.flush()
vllm_proc = subprocess.Popen(server_cmd, stdout=vllm_log_file, stderr=subprocess.STDOUT)
print(f'vLLM PID: {vllm_proc.pid}')

deadline = time.time() + 600
last_error = None
while time.time() < deadline:
    try:
        response = requests.get(f'{VLLM_BASE_URL}/models', timeout=5)
        if response.ok:
            serves_model, served_ids = models_response_serves_model(response.json(), MODEL_ID)
            if vllm_proc.poll() is not None:
                raise_vllm_startup_error(f'vLLM exited early with code {vllm_proc.returncode}')
            if serves_model:
                print(f'vLLM server is ready and serving {MODEL_ID}')
                break
            last_error = f'Healthy server is not serving {MODEL_ID}; served models: {served_ids or "unknown"}'
        else:
            last_error = f'HTTP {response.status_code}: {response.text[:200]}'
    except Exception as exc:
        last_error = repr(exc)
    if vllm_proc.poll() is not None:
        raise_vllm_startup_error(f'vLLM exited early with code {vllm_proc.returncode}')
    time.sleep(5)
else:
    raise_vllm_startup_error(f'vLLM server did not become ready: {last_error}')

In [ ]:
import requests

models_response = requests.get(f'{VLLM_BASE_URL}/models', timeout=10)
models_response.raise_for_status()
print(models_response.json())

In [ ]:
from pathlib import Path

import pandas as pd

from src.llm_classifier.constants import NLP_TYPES
from src.llm_classifier.llm_classifier import HierarchicalLLMClassifier
from src.llm_classifier.llm_classifier import build_few_shot_examples
from src.llm_classifier.prompts import build_classifier_messages
from src.utils import build_sample_id, load_config

cfg = load_config()
text_col = cfg['dataset']['text_col']
label_col = cfg['dataset']['label_col']

train_path = Path(SPLITS_DIR) / 'train.parquet'
eval_path = Path(SPLITS_DIR) / f'{SPLIT}.parquet'
if not train_path.exists():
    raise FileNotFoundError(f'Missing train split: {train_path}')
if not eval_path.exists():
    raise FileNotFoundError(f'Missing eval split: {eval_path}')

train_df = pd.read_parquet(train_path)
eval_df = pd.read_parquet(eval_path)

for path, df, required in [
    (train_path, train_df, {text_col, label_col}),
    (eval_path, eval_df, {text_col}),
]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing required columns: {sorted(missing)}')

if LIMIT is not None:
    eval_df = eval_df.head(int(LIMIT)).copy()
else:
    eval_df = eval_df.copy()

eval_df['sample_id'] = eval_df[text_col].apply(build_sample_id)
few_shot, used_ids = build_few_shot_examples(train_df, cfg)
print(f'Train rows: {len(train_df):,}; eval rows: {len(eval_df):,}; few-shot pairs: {len(few_shot)}')

In [ ]:
import json

from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key='EMPTY')
VALID_NLP_TYPES = set(NLP_TYPES) | {'none'}


def build_static_few_shot_messages(text: str) -> list[dict]:
    messages = []
    for benign_text, attack_text, attack_type in few_shot:
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{benign_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'benign',
                'confidence': 95,
                'nlp_attack_type': 'none',
                'evidence': '',
                'reason': 'No active attempt to override instructions, exfiltrate data, or hijack tools.',
            }),
        })
        if attack_type in NLP_TYPES:
            evidence = ''
            adv_reason = f'Perturbed tokens characteristic of {attack_type} adversarial attack.'
        else:
            evidence = attack_text[:80]
            adv_reason = f'Contains {attack_type} obfuscation; active adversarial prompt detected.'
        messages.append({'role': 'user', 'content': f'INPUT_PROMPT:\n{attack_text}'})
        messages.append({
            'role': 'assistant',
            'content': json.dumps({
                'label': 'adversarial',
                'confidence': 84,
                'nlp_attack_type': attack_type if attack_type in NLP_TYPES else 'none',
                'evidence': evidence,
                'reason': adv_reason,
            }),
        })
    return messages


def extract_completion_logprobs(response) -> list[dict] | None:
    return HierarchicalLLMClassifier._extract_completion_logprobs(response.model_dump())


def normalize_classifier_payload(payload: dict, raw_response_text: str | None, token_logprobs: list[dict] | None) -> dict:
    if not isinstance(payload, dict) or not payload:
        raw_response_text = raw_response_text or ''
        label = 'adversarial'
        nlp_attack_type = 'none'
        category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)
        return {
            'label': label,
            'confidence': 0.0,
            'nlp_attack_type': nlp_attack_type,
            'evidence': 'parse_failure',
            'reason': 'classifier response could not be parsed',
            '_token_logprobs': token_logprobs,
            '_provider_name': 'vllm-local',
            '_model_name': MODEL_ID,
            '_raw_response_text': raw_response_text,
            '_parse_success': False,
            '_category': category,
            'clf_label': label,
            'clf_category': category,
            'clf_confidence': 0.0,
            'clf_evidence': 'parse_failure',
            'clf_nlp_attack_type': nlp_attack_type,
            'clf_provider_name': 'vllm-local',
            'clf_model_name': MODEL_ID,
            'clf_raw_response_text': raw_response_text,
            'clf_parse_success': False,
            'clf_token_logprobs': token_logprobs,
        }

    label = payload.get('label', '') if payload else ''
    if label not in ('benign', 'adversarial', 'uncertain'):
        label = 'adversarial'

    nlp_attack_type = payload.get('nlp_attack_type', 'none') if payload else 'none'
    if nlp_attack_type not in VALID_NLP_TYPES:
        nlp_attack_type = 'none'
    if label != 'adversarial':
        nlp_attack_type = 'none'

    evidence = payload.get('evidence', '') if payload else ''
    if label != 'adversarial':
        evidence = ''

    confidence = HierarchicalLLMClassifier._normalize_confidence(
        payload.get('confidence', 50) if payload else 50
    )
    category = HierarchicalLLMClassifier._derive_category(label, nlp_attack_type)

    return {
        'label': label,
        'confidence': confidence,
        'nlp_attack_type': nlp_attack_type,
        'evidence': evidence,
        'reason': payload.get('reason', '') if payload else '',
        '_token_logprobs': token_logprobs,
        '_provider_name': 'vllm-local',
        '_model_name': MODEL_ID,
        '_raw_response_text': raw_response_text,
        '_parse_success': bool(payload),
        '_category': category,
        'clf_label': label,
        'clf_category': category,
        'clf_confidence': confidence,
        'clf_evidence': evidence,
        'clf_nlp_attack_type': nlp_attack_type,
        'clf_provider_name': 'vllm-local',
        'clf_model_name': MODEL_ID,
        'clf_raw_response_text': raw_response_text,
        'clf_parse_success': bool(payload),
        'clf_token_logprobs': token_logprobs,
    }


def parse_classifier_json(raw_response_text: str | None) -> dict:
    if raw_response_text is None:
        return {}
    try:
        payload = json.loads(raw_response_text)
        if isinstance(payload, str):
            payload = json.loads(payload)
        return payload if isinstance(payload, dict) else {}
    except json.JSONDecodeError:
        return {}


def classify_text(text: str) -> dict:
    messages = build_classifier_messages(text, build_static_few_shot_messages(text))
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=cfg['llm']['temperature'],
        max_tokens=cfg['llm']['max_tokens_classifier'],
        response_format={'type': 'json_object'},
        logprobs=True,
        top_logprobs=cfg['llm'].get('top_logprobs', 5),
    )
    raw_response_text = response.choices[0].message.content
    token_logprobs = extract_completion_logprobs(response)
    payload = parse_classifier_json(raw_response_text)
    return normalize_classifier_payload(payload, raw_response_text, token_logprobs)

In [ ]:
GROUND_TRUTH_COLUMNS = [
    'modified_sample',
    'original_sample',
    'attack_name',
    'label_binary',
    'label_category',
    'label_type',
    'prompt_hash',
    'benign_source',
    'is_synthetic_benign',
]
PREDICTION_COLUMNS = [
    'llm_pred_binary',
    'llm_pred_raw',
    'llm_pred_category',
    'llm_conf_binary',
    'llm_evidence',
    'llm_stages_run',
    'llm_provider_name',
    'llm_model_name',
    'llm_raw_response_text',
    'llm_parse_success',
    'clf_label',
    'clf_category',
    'clf_confidence',
    'clf_evidence',
    'clf_nlp_attack_type',
    'clf_provider_name',
    'clf_model_name',
    'clf_raw_response_text',
    'clf_parse_success',
    'clf_token_logprobs',
]


def valid_prediction_mask(df: pd.DataFrame) -> pd.Series:
    required_columns = {'sample_id', *PREDICTION_COLUMNS}
    if not required_columns.issubset(df.columns):
        return pd.Series(False, index=df.index)
    required_non_null_columns = ['sample_id', *PREDICTION_COLUMNS]
    return (
        df[required_non_null_columns].notna().all(axis=1)
        & df['llm_stages_run'].eq(1)
        & df['llm_pred_binary'].isin({'benign', 'adversarial'})
        & df['llm_pred_raw'].isin({'benign', 'adversarial', 'uncertain'})
        & df['llm_conf_binary'].notna()
        & df['clf_confidence'].notna()
        & df['llm_parse_success'].notna()
    )


def make_failure_result(raw_response_text: str | None = None) -> dict:
    return normalize_classifier_payload({}, raw_response_text, None)


def build_output_row(input_row: pd.Series, result: dict) -> dict:
    label = result['label']
    label_binary = 'benign' if label == 'benign' else 'adversarial'
    row = {'sample_id': input_row['sample_id']}
    for column in GROUND_TRUTH_COLUMNS:
        if column in input_row.index:
            row[column] = input_row[column]
    row.update({
        'llm_pred_binary': label_binary,
        'llm_pred_raw': label,
        'llm_pred_category': result['_category'],
        'llm_conf_binary': result['confidence'],
        'llm_evidence': result.get('evidence', ''),
        'llm_stages_run': 1,
        'llm_provider_name': result['_provider_name'],
        'llm_model_name': result['_model_name'],
        'llm_raw_response_text': result['_raw_response_text'],
        'llm_parse_success': result['_parse_success'],
        'clf_label': label,
        'clf_category': result['_category'],
        'clf_confidence': result['confidence'],
        'clf_evidence': result.get('evidence', ''),
        'clf_nlp_attack_type': result['nlp_attack_type'],
        'clf_provider_name': result['_provider_name'],
        'clf_model_name': result['_model_name'],
        'clf_raw_response_text': result['_raw_response_text'],
        'clf_parse_success': result['_parse_success'],
        'clf_token_logprobs': json.dumps(result.get('_token_logprobs')),
    })
    return row


def write_checkpoint(rows: list[dict]) -> None:
    if not rows:
        return
    checkpoint = Path(CHECKPOINT_PATH)
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    new_df = pd.DataFrame(rows)
    if checkpoint.exists():
        existing_df = pd.read_parquet(checkpoint)
        out_df = pd.concat([existing_df, new_df], ignore_index=True)
        out_df = out_df.drop_duplicates(subset=['sample_id'], keep='last')
    else:
        out_df = new_df
    out_df.to_parquet(checkpoint, index=False)
    print(f'Checkpoint rows: {len(out_df):,} -> {checkpoint}')


In [ ]:
from tqdm.auto import tqdm

completed_ids = set()
checkpoint_path = Path(CHECKPOINT_PATH)
if checkpoint_path.exists():
    checkpoint_df = pd.read_parquet(checkpoint_path)
    valid_checkpoint_df = checkpoint_df[valid_prediction_mask(checkpoint_df)].copy()
    invalid_checkpoint_rows = len(checkpoint_df) - len(valid_checkpoint_df)
    if invalid_checkpoint_rows:
        print(f'Ignoring invalid checkpoint rows: {invalid_checkpoint_rows:,}')
    completed_ids = set(valid_checkpoint_df['sample_id'].tolist()) if 'sample_id' in valid_checkpoint_df.columns else set()
    print(f'Resuming from checkpoint: {len(completed_ids):,} completed rows')

pending_df = eval_df[~eval_df['sample_id'].isin(completed_ids)].reset_index(drop=True)
print(f'Pending rows: {len(pending_df):,}')

buffer: list[dict] = []
for idx, input_row in tqdm(list(pending_df.iterrows()), total=len(pending_df), desc=f'Classifying {SPLIT}'):
    text = str(input_row[text_col])
    try:
        result = classify_text(text)
    except Exception as exc:
        try:
            health = requests.get(f'{VLLM_BASE_URL}/models', timeout=5)
            health.raise_for_status()
        except Exception as health_exc:
            raise RuntimeError(f'vLLM server is unreachable after row failure: {health_exc}') from exc
        print(f'Row failed but server is healthy; writing parse-failure row for sample_id={input_row.sample_id}: {exc}')
        result = make_failure_result(raw_response_text=None)
    buffer.append(build_output_row(input_row, result))
    if len(buffer) >= CHECKPOINT_EVERY:
        write_checkpoint(buffer)
        buffer.clear()

write_checkpoint(buffer)

if checkpoint_path.exists():
    final_df = pd.read_parquet(checkpoint_path)
else:
    final_df = pd.DataFrame(columns=['sample_id'] + PREDICTION_COLUMNS)

pre_filter_rows = len(final_df)
final_df = final_df[valid_prediction_mask(final_df)].copy()
invalid_final_rows = pre_filter_rows - len(final_df)
if invalid_final_rows:
    print(f'Dropped invalid final rows: {invalid_final_rows:,}')
final_df = final_df.drop_duplicates(subset=['sample_id'], keep='last')
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(OUTPUT_PATH, index=False)
print(f'Final rows: {len(final_df):,} -> {OUTPUT_PATH}')


In [ ]:
out_df = pd.read_parquet(OUTPUT_PATH)
expected_columns = {'sample_id', *PREDICTION_COLUMNS}
missing_columns = expected_columns - set(out_df.columns)
if missing_columns:
    raise AssertionError(f'Missing expected columns: {sorted(missing_columns)}')
assert not any(col.startswith('judge_') for col in out_df.columns), 'Output must not contain judge_* columns'
assert valid_prediction_mask(out_df).all(), 'Output contains invalid classifier prediction rows'
assert set(out_df['llm_stages_run'].dropna().unique()) == {1}, 'Classifier-only outputs must have llm_stages_run=1'
print(f'Read back {len(out_df):,} rows from {OUTPUT_PATH}')
print(out_df['llm_pred_binary'].value_counts(dropna=False))
print(out_df[['llm_pred_raw', 'llm_conf_binary', 'llm_parse_success']].head())
